Таким форматом в Pandas является формат datetime, который записывается как YYYY-MM-DD HH: MM: SS, то есть составляющие времени указываются в следующем порядке: год, месяц, день, час, минута, секунда.

In [1]:
import pandas as pd
melb_data = pd.read_csv('data/melb_data_ps.csv')

# display(melb_data.head())
display(melb_data['Date'])


0         3/12/2016
1         4/02/2016
2         4/03/2017
3         4/03/2017
4         4/06/2016
            ...    
13575    26/08/2017
13576    26/08/2017
13577    26/08/2017
13578    26/08/2017
13579    26/08/2017
Name: Date, Length: 13580, dtype: str

Для того чтобы преобразовывать столбцы с датами, записанными в распространённых форматах, в формат datetime, можно воспользоваться функцией pandas.to_datetime(). В нашем случае в функции нужно указать параметр dayfirst=True, который будет обозначать, что в первоначальном признаке первым идет день. Преобразуем столбец Date в формат datetime, передав его в эту функцию:

In [2]:
melb_data['Date'] = pd.to_datetime(melb_data['Date'], dayfirst=True)
display(melb_data['Date'])

0       2016-12-03
1       2016-02-04
2       2017-03-04
3       2017-03-04
4       2016-06-04
           ...    
13575   2017-08-26
13576   2017-08-26
13577   2017-08-26
13578   2017-08-26
13579   2017-08-26
Name: Date, Length: 13580, dtype: datetime64[us]

## Выделение атрибутов datetime

Тип данных datetime позволяет с помощью специального аксессора dt выделять составляющие времени из каждого элемента столбца, такие как:

- date — дата;
- year, month, day — год, месяц, день;
- time — время;
- hour, minute, second — час, минута, секунда;
- dayofweek — номер дня недели, от 0 до 6, где 0 — понедельник, 6 — воскресенье;
- day_name — название дня недели;
- dayofyear — порядковый день года;
- q uarter — квартал (интервал в три месяца).


Например, обратившись по атрибуту dt.year в столбце Date, мы можем «достать» год продажи и понять, за какой интервал времени (в годах) представлены наши данные, а также на какой год приходится наибольшее число продаж:

In [3]:
years_sold  = melb_data['Date'].dt.year
print(years_sold)
print('min:', years_sold.min())
print('max:', years_sold.max())
print('mode:', years_sold.mode()[0])


0        2016
1        2016
2        2017
3        2017
4        2016
         ... 
13575    2017
13576    2017
13577    2017
13578    2017
13579    2017
Name: Date, Length: 13580, dtype: int32
min: 2016
max: 2017
mode: 2017


Теперь попробуем понять, на какие месяцы приходится пик продаж объектов недвижимости. Для этого выделим атрибут dt.month и на этот раз занесём результат в столбец MonthSale, а затем найдём относительную частоту продаж для каждого месяца от общего количества продаж — для этого используем метод value_counts() с параметром normalize (вывод в долях):

In [4]:
melb_data['MonthSale'] = melb_data['Date'].dt.month
melb_data['MonthSale'].value_counts(normalize=True)



MonthSale
5     0.149411
7     0.145950
9     0.135862
6     0.134757
8     0.114138
11    0.082032
4     0.069882
3     0.049926
12    0.044698
10    0.040574
2     0.032622
1     0.000147
Name: proportion, dtype: float64

Из результатов становится ясно, что наибольшее количество продаж недвижимости приходится на май, июль и сентябрь (пятый, седьмой и девятый месяцы соответственно). Месяцами застоя при этом являются месяцы — октябрь, февраль и январь (десятый, второй и первый месяцы соответственно).

## Работа с интервалами

Часто бывает такая ситуация, что необходимо вычислять интервалы между двумя временными промежутками. Например, можно вычислить, сколько дней прошло с 1 января 2016 года до момента продажи объекта. Для этого можно просто найти разницу между датами продаж и заявленной датой, представленной в формате datetime: 

In [5]:
delta_days = melb_data['Date'] - pd.to_datetime('2016-01-01')
print(delta_days)

0       337 days
1        34 days
2       428 days
3       428 days
4       155 days
          ...   
13575   603 days
13576   603 days
13577   603 days
13578   603 days
13579   603 days
Name: Date, Length: 13580, dtype: timedelta64[us]


В результате мы получаем Series, элементами которой является количество дней, которое прошло с 1 января 2016 года. Обратите внимание, что данные такого формата относятся к типу timedelta.

img
Рассмотрим другой пример. Давайте создадим признак возраста объекта недвижимости в годах на момент продажи. Для этого выделим из столбца с датой продажи год и вычтем из него год постройки здания. Результат оформим в виде столбца AgeBuilding:

In [6]:
melb_data['ageBuliding'] = melb_data['Date'].dt.year - melb_data['YearBuilt']
melb_data.drop('YearBuilt', axis=1)

,index,Suburb,Address,Rooms,Type,Price,Method,SellerG,Date,Distance,...,Landsize,BuildingArea,CouncilArea,Lattitude,Longtitude,Regionname,Propertycount,Coordinates,MonthSale,ageBuliding
0,0,Abbotsford,85 Turner St,2,h,1480000.0,S,Biggin,2016-12-03,2.5,...,202.0,126.0,Yarra,-37.79960,144.99840,Northern Metropolitan,4019,"-37.7996, 144.9984",12,46
1,1,Abbotsford,25 Bloomburg St,2,h,1035000.0,S,Biggin,2016-02-04,2.5,...,156.0,79.0,Yarra,-37.80790,144.99340,Northern Metropolitan,4019,"-37.8079, 144.9934",2,116
2,2,Abbotsford,5 Charles St,3,h,1465000.0,SP,Biggin,2017-03-04,2.5,...,134.0,150.0,Yarra,-37.80930,144.99440,Northern Metropolitan,4019,"-37.8093, 144.9944",3,117
3,3,Abbotsford,40 Federation La,3,h,850000.0,PI,Biggin,2017-03-04,2.5,...,94.0,126.0,Yarra,-37.79690,144.99690,Northern Metropolitan,4019,"-37.7969, 144.9969",3,47
4,4,Abbotsford,55a Park St,4,h,1600000.0,VB,Nelson,2016-06-04,2.5,...,120.0,142.0,Yarra,-37.80720,144.99410,Northern Metropolitan,4019,"-37.8072, 144.9941",6,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13575,13575,Wheelers Hill,12 Strada Cr,4,h,1245000.0,S,Barry,2017-08-26,16.7,...,652.0,126.0,NaN,-37.90562,145.16761,South-Eastern Metropolitan,7392,"-37.90562, 145.16761",8,36
13576,13576,Williamstown,77 Merrett Dr,3,h,1031000.0,SP,Williams,2017-08-26,6.8,...,333.0,133.0,NaN,-37.85927,144.87904,Western Metropolitan,6380,"-37.85927, 144.87904",8,22
13577,13577,Williamstown,83 Power St,3,h,1170000.0,S,Raine,2017-08-26,6.8,...,436.0,126.0,NaN,-37.85274,144.88738,Western Metropolitan,6380,"-37.85274, 144.88738",8,20
13578,13578,Williamstown,96 Verdon St,4,h,2500000.0,PI,Sweeney,2017-08-26,6.8,...,866.0,157.0,NaN,-37.85908,144.89299,Western Metropolitan,6380,"-37.85908, 144.89299",8,97


Создайте в таблице melb_df признак WeekdaySale (день недели). Найдите, сколько объектов недвижимости было продано в выходные (суббота и воскресенье), результат занесите в переменную weekend_count. В качестве ответа введите результат вывода переменной weekend_count.

In [7]:
melb_data['WeekdaySale'] = melb_data['Date'].dt.dayofweek
#isin - есть ли в. Проверка на наличие елемента в Df выдает булевые ответы True  \ False. Shape[0] - считает их
weekend_count = melb_data[melb_data['WeekdaySale'].isin([5,6])].shape[0]
print(weekend_count)


12822


Вам представлены данные (в формате csv) об отчётах очевидцев НЛО в США за период с 1930 по 2020 год.

В данных есть следующие признаки:

- "City" — город, где был замечен НЛО;
- "Colors Reported" — цвет объекта;
- "Shape Reported" — форма объекта;
- "State" — обозначение штата;
- "Time" — время, когда был замечен НЛО (данные отсортированы от старых наблюдений к новым). 
Прочитайте данные, сделайте преобразование времени к формату datetime и выполните задания ниже.

In [8]:
ufo_data = pd.read_csv('data/ufo.csv', sep=',')
# melb_data['Date'] = pd.to_datetime(melb_data['Date'], dayfirst=True)
ufo_data['Time'] = pd.to_datetime(ufo_data['Time'])
print(ufo_data['Time'].dt.year.mode())
ufo_data.head(20)

0    1999
Name: Time, dtype: int32


,City,Colors Reported,Shape Reported,State,Time
0,Ithaca,NaN,TRIANGLE,NY,1930-06-01 22:00:00
1,Willingboro,NaN,OTHER,NJ,1930-06-30 20:00:00
2,Holyoke,NaN,OVAL,CO,1931-02-15 14:00:00
3,Abilene,NaN,DISK,KS,1931-06-01 13:00:00
4,New York Worlds Fair,NaN,LIGHT,NY,1933-04-18 19:00:00
5,Valley City,NaN,DISK,ND,1934-09-15 15:30:00
6,Crater Lake,NaN,CIRCLE,CA,1935-06-15 00:00:00
7,Alma,NaN,DISK,MI,1936-07-15 00:00:00
8,Eklutna,NaN,CIGAR,AK,1936-10-15 17:00:00
9,Hubbard,NaN,CYLINDER,OR,1937-06-15 00:00:00


Найдите средний интервал времени (в днях) между двумя последовательными случаями наблюдения НЛО в штате Невада (NV).
Чтобы выделить дату из столбца Time, можно воспользоваться атрибутом datetime date.

Чтобы вычислить разницу между двумя соседними датами в столбце, примените к нему метод diff().

Чтобы перевести интервал времени в дни, воспользуйтесь атрибутом timedelta days.

In [9]:
# отфильтровывам ниваду
nv_data = ufo_data[ufo_data['State'] == 'NV'].copy()
# берем только дату 
nv_data['Date'] = nv_data['Time'].dt.date
# считаем разницу
nv_data['Diff'] = nv_data['Date'].diff()
# перводим в дни
nv_data['diffDays'] = nv_data['Diff'].dt.days
# среднее
print(round(nv_data['diffDays'].mean()))
display(nv_data)

69


,City,Colors Reported,Shape Reported,State,Time,Date,Diff,diffDays
76,Las Vegas,NaN,DISK,NV,1947-07-15 10:00:00,1947-07-15,NaT,NaN
172,Nellis AFB,NaN,DISK,NV,1952-02-17 18:00:00,1952-02-17,1678 days,1678.0
565,Fallon,NaN,OVAL,NV,1959-09-15 00:00:00,1959-09-15,2767 days,2767.0
566,Goldfield,NaN,LIGHT,NV,1959-09-15 01:00:00,1959-09-15,0 days,0.0
613,NaN,NaN,DISK,NV,1960-07-01 12:00:00,1960-07-01,290 days,290.0
...,...,...,...,...,...,...,...,...
17447,Laughlin,NaN,FORMATION,NV,2000-09-16 22:00:00,2000-09-16,22 days,22.0
17567,Las Vegas,NaN,SPHERE,NV,2000-09-30 22:25:00,2000-09-30,14 days,14.0
17617,Las Vegas,RED YELLOW,OTHER,NV,2000-10-06 20:25:00,2000-10-06,6 days,6.0
17890,Reno,NaN,TRIANGLE,NV,2000-11-07 02:15:00,2000-11-07,32 days,32.0
